## RAG - Chunking &amp; Splitting
In this notebook, we'll cover a very important aspect of RAG - chunking and splitting. One of the most effective strategies for improving the performance of your LLMs is to split large documents into smaller chunks. The goal is to give your LLM only the information that it needs for your task _and nothing more_. This practice is the art & science of _text splitting_. This is one of the first and most foundational decisions that a language model practitioner needs to make. 

In this notebook, we'll cover _5 levels of text splitting_ that can squeeze out more performance from your LLM applications using the same data that you already have.
1. By physical positionioning & structure of text chunks
    * Character Splitting - splitting your document by a static character limit
    * Recursive Character Text Splitting - recursively go through your document and split it by list of separators (e.g. whitespace characters)
    * Document Specific Text splitting - for example, use different techniques for different types of documents (JavaDocs, PythonDocs, PDFs including embedded images - multimodality handling)
2. Splitting that does not rely on just structure, but the what and the why of text (i.e. meaning & context of chunks)
    * Semantic Splitting 

Application of LLMs are better when you given them your own data, rather than rely only on the pre-training data. Given the context window limitation of most models (recent models such as Google Gemini being an exception!), in most cases you cannot feed the contents of your entire model to the LLM. Secondly, LLMs always do better when to increase the _signal to noise_ ratio (i.e. remove information that isn't helpful to your task (noise)) - distracting information in the model's context tends to _measurably destroy performance_ of LLM. So you want to prune the fluff from your data wherever possible.

There is no single technique that can fit into all requirements - you need to use sevaral techniques & evaluate outcomes to decide what fits best.

See: [Full Stack Retrieval](fullstackretrieval.com)



In [1]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from IPython.display import display, Markdown

from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# since we are using Gemini, we'll use Google embeddings
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# I seem to have permanently exhausted rate limt on Google embeddings on free tier,
# I don't want to enable billing, so am switching to Cohere embeddings
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

In [2]:
# load API keys from .env files
load_dotenv(override=True)
console = Console()

In [3]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
faiss_store = pathlib.Path(os.getcwd()) / "faiss_index_rag_qd"

### Logical Routing

A business scenario that could explain the need for Logical routing would be something like the following: Let's say WeCoverAnything (WCA) is an Insurance company tha sells Life Insurance, Health Insurance and Car Insurance policies. Bob is a client who has bought both Health and Car insurnace policies from WCA. WCA has deployed a customer facing chatbot (powered by an LLM of course!) that can answer common questions from end-customers (or redirect query to Helpdesk agents in case it cannot). Bob could ask a question like "Is my policy covering both own damage and third-party liability? Can you explain what each one means?" or something like "I need to file a claim for a recent hospitalization. What documents do I need, and how do I submit them?". Clearly the first relates to Car Insurance and the second to Health Insurance. The chabot must be intelligent to direct the former question to "Car Insurance Policy" datasources and the latter to "Health Insurance Policy" data sources.

Let's walk through a Logical Routing use-case. Suppose that we have 3 knowledge sources focused related to Python, JavaScript and Go programming respectively. So, we'd like to re-direct all Python queries to the Python datastore, JavaScript to the JS data store and so on. 

<center>
<img src="images/logical_routing.png"/>
</center>

We bind LLM output to a structure (a Pydantic datamodel), so that it returns one of a fixed set of outputs, which we can then use to redirect to specific datasource. We use the `llm_with_structured_output(...)` call to bind LLM's output, similar to what we did in the [Classification Example](04_classificaltion.py)

In [4]:
from typing import Literal
from pydantic import BaseModel, Field


# Data model: here we define various "routes" depending on programming language
# So Python related queries should go to "python_docs", JavaScript to "js_docs" etc.
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    # these could be paths of vector stores, or identifiers for APIs etc.
    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question choose which datasource would be most relevant for answering their question",
    )


structured_llm = llm.with_structured_output(RouteQuery)

In [5]:
# Prompt
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# Define router
router = prompt | structured_llm

In [7]:
# now let's try it out with various languages
# Python first
question_python = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question_python})
print(result.datasource)

python_docs


In [8]:
# How about this?
question_go = """Why doesn't the following code work:

import (
	"fmt"
)

func main() {
	m := make(map[string]int)
	vals := []int{1, 2, 3}

	for _, v := range vals {
		go func() {
			m["sum"] += v
		}()
	}

	fmt.Println("sum:", m["sum"])
}
"""

result = router.invoke({"question": question_go})
print(result.datasource)

golang_docs


In [9]:
# And this?
question_js = """Why doesn't the following code work:

let count = 0;

for (var i = 0; i < 5; i++) {
  setInterval(function () {
    count++;
    console.log("i:", i, "count:", count);
    if (count === 5) {
      clearInterval(this);
    }
  }, 1000);
}
"""

result = router.invoke({"question": question_js})
print(result.datasource)

js_docs


So as you can see, the LLM is able to _detect_ the programming language and _direct_ us to the correct data source to redirect our query to!

Now let us create a common function to route.

In [10]:
def choose_route(result):
    if "python_docs" in result.datasource.lower():
        ### Logic here
        return "chain for python_docs"
    elif "js_docs" in result.datasource.lower():
        ### Logic here
        return "chain for js_docs"
    else:
        ### Logic here
        return "golang_docs"


from langchain_core.runnables import RunnableLambda

full_chain = router | RunnableLambda(choose_route)

In [11]:
full_chain.invoke({"question": question_js})

'chain for js_docs'

In [12]:
full_chain.invoke({"question": question_python})

'chain for python_docs'

In [13]:
full_chain.invoke({"question": question_go})

'golang_docs'

### Semantic Routing

Semantic routing is a little bit straightforward as compared to Logical Routing. In this case we have a series of prompts (or more correctly, prompt templates) we want to choose depending on the input query - so say. we have a Physics related prompt-template and a Math related prompt-template and the user asks a question related to either Physics or Maths. 

First we embed both the templates as well as the user's question. Then we use a `cosine similarity` function to choose which subject the prompt is related to, and then fire the "closest match" prompt.

<center>
<img src="images/semantic_routing.png"/>
</center>

In [14]:
from langchain.utils.math import cosine_similarity
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [15]:
# Two prompt templates for different subjects
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

In [16]:
embeddings = CohereEmbeddings(
    model="embed-english-v3.0",
)
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

In [17]:
# Route question to correct prompt
def prompt_router(input):
    # Embed question
    query_embedding = embeddings.embed_query(input["query"])
    # Compute similarity between embedded query and embedded prompt templates
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    print(f"Similarities -> {similarity}")
    most_similar = prompt_templates[similarity.argmax()]
    # Chosen prompt
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)

In [18]:
prompt_router({"query": "What is Newton's second law of motion?"})  # a Physics question

Similarities -> [0.27495906 0.18490319]
Using PHYSICS


PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template="You are a very smart physics professor. You are great at answering questions about physics in a concise and easy to understand manner. When you don't know the answer to a question you admit that you don't know.\n\nHere is a question:\n{query}")

In [19]:
# now let's build our chain
chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | llm
    | StrOutputParser()
)

In [22]:
chain.invoke("What is Newton's second law of motion?")  # Physics question

Similarities -> [0.27495906 0.18490319]
Using PHYSICS


"Newton's second law of motion states that the acceleration of an object is directly proportional to the net force acting on it and inversely proportional to its mass.\n\nIn simpler terms, it's often expressed by the famous equation:\n\n**F = ma**\n\nWhere:\n*   **F** is the net force acting on the object (measured in Newtons).\n*   **m** is the mass of the object (measured in kilograms).\n*   **a** is the acceleration of the object (measured in meters per second squared).\n\nThis means that if you apply a larger net force to an object, it will accelerate more. If the object has more mass, it will accelerate less for the same amount of force. The acceleration always occurs in the same direction as the net force."

In [23]:
chain.invoke("What is Pythagoras' theorem?")  # Math question

Similarities -> [0.20418755 0.2830592 ]
Using MATH


'Pythagoras\' theorem is a fundamental principle in geometry that describes the relationship between the three sides of a **right-angled triangle**.\n\nLet\'s break it down:\n\n### Component 1: The Type of Triangle\n\n*   **It applies *only* to right-angled triangles.** A right-angled triangle is a triangle that has one angle measuring exactly 90 degrees (a "right angle").\n\n### Component 2: The Names of the Sides\n\nIn a right-angled triangle, the sides have specific names:\n\n*   **Hypotenuse (c):** This is the longest side of the triangle, and it is always the side directly opposite the right angle.\n*   **Legs (a and b):** These are the two shorter sides that form the right angle. They are sometimes also called "cathetus."\n\n### Component 3: The Relationship Between the Sides\n\nThe theorem states that:\n\n**"The square of the length of the hypotenuse (c) is equal to the sum of the squares of the lengths of the other two sides (a and b)."**\n\n### Putting It Together: The Formula

In [26]:
# Math or Physics?
chain.invoke(
    "If a car travels at a constant speed of 60 miles per hour, how far will it travel in 3 hours?"
)

Similarities -> [0.11114148 0.09468728]
Using PHYSICS


"That's a straightforward one!\n\nIf a car travels at a constant speed of 60 miles per hour for 3 hours, it will travel:\n\nDistance = Speed × Time\nDistance = 60 mph × 3 hours\n**Distance = 180 miles**"

In [27]:
chain.invoke(
    "What is the maximum volume of a sphere that can be contained within a cube of side length $L$, and how does the pressure exerted by an ideal gas inside that sphere relate to its temperature?"
)

Similarities -> [0.15424759 0.14749195]
Using PHYSICS


"Alright, let's break this down.\n\n1.  **Maximum Volume of the Sphere:**\n    For a sphere to be contained within a cube of side length $L$, its diameter must be no larger than $L$. To maximize the sphere's volume, its diameter must be exactly $L$.\n    So, the radius of the sphere, $R = L/2$.\n    The volume of a sphere is given by $V = \\frac{4}{3}\\pi R^3$.\n    Substituting $R = L/2$:\n    $V = \\frac{4}{3}\\pi \\left(\\frac{L}{2}\\right)^3 = \\frac{4}{3}\\pi \\frac{L^3}{8} = \\frac{\\pi L^3}{6}$.\n\n2.  **Pressure of an Ideal Gas and Temperature:**\n    For an ideal gas, the relationship between pressure ($P$) and absolute temperature ($T$) is given by the Ideal Gas Law: $PV = nRT$.\n    Here, $V$ is the volume of the gas (which is the sphere's volume), $n$ is the number of moles of gas, and $R$ is the ideal gas constant.\n    If the number of moles of gas ($n$) and the volume ($V$) are kept constant, then the pressure is directly proportional to the absolute temperature:\n    $P

In [28]:
from pydantic import BaseModel, Field


class InputQuestionsList(BaseModel):
    """Input schema for list of questions."""

    num_questions: int = Field(..., description="Number of user questions.")
    questions: list[str] = Field(..., description="List of user questions.")


structured_llm = llm.with_structured_output(InputQuestionsList)

In [31]:
prompt_template = """
From the following user input, extract the number questions asked, and extract the questions separately. User could ask multiple questions in a single input.

User Input: {user_input}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert at extracting questions from user input."),
        ("human", prompt_template),
    ]
)

question_chain = prompt | structured_llm

In [34]:
response = question_chain.invoke(
    {
        "user_input": "What is Newton's second law of motion? Also, what is Pythagoras' theorem?"
    }
)
print(f"I extracted {response.num_questions} questions.")
print("Here are the questions:")
for i in range(response.num_questions):
    print(f"{i+1} -> {response.questions[i]}")

I extracted 2 questions.
Here are the questions:
1 -> What is Newton's second law of motion?
2 -> what is Pythagoras' theorem?
